## N-glycosylation from a GLYCAM06 named PDB file (retrieved from [GlyGen](https://www.glygen.org/glycan-search/))
### Joseph R. Laforet Jr.

Three glycans retrieved from GlyGen, each ending in a hydroxyl residue (`ROH`)
on the anomeric carbon. `fragment_from_pdb` reads each file with its
residues intact. `attach` bonds the anomeric carbon to ND2 of asparagine 60
of ubiquitin. Naming `O1` as the fragment's leaving atom removes the whole
hydroxyl, so `ROH` disappears and the GLYCAM residue names survive into the
written file, which is what a force field like GLYCAM06 keys on.

In [ ]:
import logging, warnings
from rdkit import RDLogger

warnings.filterwarnings("ignore")
logging.disable(logging.WARNING)
RDLogger.DisableLog("rdApp.*")

In [ ]:
from collections import Counter
from pathlib import Path

from mbuild.biopolymers import Protein, draw_fragment, fragment_from_pdb

mbuild_glycan = fragment_from_pdb("../glycans/glycam_G57321FI.pdb")
print([(residue.name, residue.resnum) for residue in mbuild_glycan.children], mbuild_glycan.n_particles, "atoms")
print("bond orders:", dict(Counter(d["bond_order"] for *_, d in mbuild_glycan.bonds(return_bond_order=True))))

`attach` names the glycan's atoms, and a PDB file's names are the builder's, not
ours. `draw_fragment` shows them: every atom labelled, one tint per residue with
a legend, and the names we pass in red. The anomeric carbon is `C1` of the sugar
residue (residue 2), and the hydroxyl that leaves is `O1` of the `ROH` residue with
its hydrogen `HO1`. Those three names are what the `attach` call below uses.

In [ ]:
draw_fragment(mbuild_glycan, highlight=["C1", "O1"], size=(900, 560))

Asparagine's amide nitrogen is neutral, so unlike a lysine it needs no
`deprotonate` first. `attach` removes one ND2 hydrogen itself.

In [ ]:
mbuild_protein = Protein("../1ubq_protonated.pdb")
reactive_ASN = mbuild_protein.get_residue(60)
draw_fragment(reactive_ASN, highlight=["ND2"], size=(900, 560))

In [ ]:
mbuild_protein = Protein("../1ubq_protonated.pdb")
mbuild_protein.attach(
    mbuild_glycan,
    fragment_atom_name="C1", # This is the atom of the fragment that bonds to the protein
    fragment_resnum=2,
    resnum=60,
    atom_name="ND2", # This is the atom where a new bond is formed
    leaving_atom_names="HD22", # This is the atom of the protein that leaves, select from above image
    fragment_leaving_atom_names="O1", # This is the leaving atom of the fragment
)
bond_record, = mbuild_protein.bond_records()
bond_record

In [ ]:
print([(r.name, r.resnum, r.hetatm) for r in mbuild_protein.residues()][75:])
print(mbuild_protein.n_particles, "atoms: 1231 protein + 30 glycan - HD22 - O1 - HO1 =", 1231 + 30 - 3)

The same call for all three glycans, checking the written file each time.

In [ ]:
out = Path("../assets_cache")

out_files = []

for name in ("glycam_G57321FI", "glycam_G42666HT", "glycam_G15407YE"):
    mbuild_glycan = fragment_from_pdb(f"../glycans/{name}.pdb")
    mbuild_protein = Protein("../1ubq_protonated.pdb")
    
    mbuild_protein.attach(mbuild_glycan, 
                          fragment_atom_name="C1",         # This is the atom of the fragment that bonds to the protein
                          fragment_resnum=2, 
                          resnum=60, 
                          atom_name="ND2",                 # This is the atom where a new bond is formed
                          leaving_atom_names="HD22",       # This is the atom of the protein that leaves, select from draw_fragment()
                          fragment_leaving_atom_names="O1" # This is the leaving atom of the fragment
                          )

    written = out / f"1ubq_{name}.pdb"
    out_files.append(written)
    mbuild_protein.save_pdb(written, overwrite=True)
    lines = written.read_text().splitlines()
    hetero = sorted({(line[17:20], int(line[22:26])) for line in lines if line.startswith("HETATM")}, key=lambda t: t[1])
    in_file = [(r.name, r.resnum) for r in mbuild_glycan.children if r.name != "ROH"]
    print(f"{name}: {[r.name for r in mbuild_glycan.children]} -> file has {hetero}, ROH present: {'ROH' in written.read_text()}")
    assert [name for name, _ in hetero] == [name for name, _ in in_file]

Make sure you inspect the output PDB files!

In [ ]:
import nglview

view = nglview.show_file(str(out_files[2]))
view.clear_representations()
view.add_representation("cartoon", color="#990000")
view.add_representation("licorice", selection="[0VA] or [4YB] or [0MB] or 60:A)", color="#FF7733")
view.center(selection="[0VA] or [4YB] or [0MB]")
view

Every atom of the product carries a bond order and a formal charge, so it exports to RDKit and OpenFF.

In [ ]:
from openff.toolkit import Molecule

openff_molecule = Molecule.from_rdkit(mbuild_protein.to_rdkit(), allow_undefined_stereo=True)
print(openff_molecule.n_atoms, "atoms, net charge", openff_molecule.total_charge)